#1. Setup

In [ ]:
import os
os.chdir("/content")

In [ ]:
!rm -rf /content/Data
!git clone https://github.com/KU-EBL/Data.git

In [ ]:
import os
os.chdir("/content/Data/EDS")
os.listdir()

#2. Load the data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, GridSearchCV

In [ ]:
df = pd.read_csv("gradient_boosting_environment_practice.csv")
display(df.head())
print("Shape:", df.shape)
print(df.dtypes)

# 3. Data preprocessing

In [ ]:
df_encoded = pd.get_dummies(
    df,
    columns=["TN_status"],
    dtype=int
)

display(df_encoded.head())

print("Original columns:")
print(df.columns.tolist())

print("\nColumns after one-hot encoding:")
print(df_encoded.columns.tolist())

In [ ]:
feature_cols = [
    "Flow_m3_s",
    "Temperature_C",
    "pH",
    "DO_mg_L",
    "Turbidity_NTU",
    "EC_uS_cm",
    "TN_status_Good"
]

X = df_encoded[feature_cols]
y = df_encoded["TOC_mg_L"]

In [ ]:
# First split: 80% development data and 20% test data
X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# Second split: development data into 75% training and 25% validation
# This produces 60% train, 20% validation, and 20% test overall.
X_train, X_val, y_train, y_val = train_test_split(
    X_dev,
    y_dev,
    test_size=0.25,
    random_state=42
)

print("Training samples  :", len(X_train))
print("Validation samples:", len(X_val))
print("Test samples      :", len(X_test))

# 4. Apply GB

In [ ]:
baseline_model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=2,
    min_samples_leaf=5,
    subsample=1.0,
    loss="squared_error",
    random_state=42
)

baseline_model.fit(X_train, y_train)

# Make predictions on the validation set
y_val_pred = baseline_model.predict(X_val)

# Evaluate the baseline model
val_mae = mean_absolute_error(y_val, y_val_pred)
val_rmse = mean_squared_error(y_val, y_val_pred) ** 0.5
val_r2 = r2_score(y_val, y_val_pred)

print("Baseline model performance on the validation set")
print(f"MAE  : {val_mae:.3f}")
print(f"RMSE : {val_rmse:.3f}")
print(f"R²   : {val_r2:.3f}")


In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(y_val, y_val_pred, alpha=0.75)

minimum = min(y_val.min(), y_val_pred.min())
maximum = max(y_val.max(), y_val_pred.max())

# Draw the 1:1 reference line
plt.plot(
    [minimum, maximum],
    [minimum, maximum],
    linestyle="--"
)

plt.xlabel("Observed TOC (mg/L)")
plt.ylabel("Predicted TOC (mg/L)")
plt.title("Observed vs. Predicted: Validation Set")
plt.tight_layout()
plt.show()

In [ ]:
learning_rates = [0.01, 0.05, 0.1, 0.2]
results = []

for lr in learning_rates:
    comparison_model = GradientBoostingRegressor(
        n_estimators=150,
        learning_rate=lr,
        max_depth=2,
        min_samples_leaf=5,
        random_state=42
    )

    # Train using the training set
    comparison_model.fit(X_train, y_train)

    # Evaluate using the validation set
    val_pred = comparison_model.predict(X_val)

    results.append({
        "learning_rate": lr,
        "MAE": mean_absolute_error(y_val, val_pred),
        "RMSE": mean_squared_error(y_val, val_pred) ** 0.5,
        "R2": r2_score(y_val, val_pred)
    })

learning_rate_results = (
    pd.DataFrame(results)
    .sort_values("RMSE")
    .reset_index(drop=True)
)

display(learning_rate_results)

In [ ]:
# Visualize validation RMSE for each learning rate
plt.figure(figsize=(6, 4))

plt.scatter(
    learning_rate_results["learning_rate"],
    learning_rate_results["RMSE"],
    marker="o"
)

plt.xlabel("Learning Rate")
plt.ylabel("Validation RMSE")
plt.title("Validation RMSE by Learning Rate")
plt.tight_layout()
plt.show()

#5. Hyperparameter Optimization

In [ ]:
# Define the Grid Search parameter combinations
param_grid = {
    "n_estimators": [100, 150, 200],
    "learning_rate": [0.03, 0.05, 0.1],
    "max_depth": [1, 2, 3],
    "min_samples_leaf": [3, 5, 10],
    "subsample": [0.8, 1.0]
}

grid_search = GridSearchCV(
    estimator=GradientBoostingRegressor(
        loss="squared_error",
        random_state=42
    ),
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("Best parameters:")
print(grid_search.best_params_)

print("\nBest cross-validation RMSE:")
print(-grid_search.best_score_)


# Evaluate the best Grid Search model
best_grid_model = grid_search.best_estimator_
grid_pred = best_grid_model.predict(X_test)

grid_mae = mean_absolute_error(y_test, grid_pred)
grid_rmse = mean_squared_error(y_test, grid_pred) ** 0.5
grid_r2 = r2_score(y_test, grid_pred)

print(f"Grid Search MAE  : {grid_mae:.3f}")
print(f"Grid Search RMSE : {grid_rmse:.3f}")
print(f"Grid Search R²   : {grid_r2:.3f}")


# Compare baseline and tuned models
baseline_pred = baseline_model.predict(X_test)

comparison = pd.DataFrame({
    "Model": ["Baseline", "Grid Search"],
    "MAE": [
        mean_absolute_error(y_test, baseline_pred),
        mean_absolute_error(y_test, grid_pred)
    ],
    "RMSE": [
        mean_squared_error(y_test, baseline_pred) ** 0.5,
        mean_squared_error(y_test, grid_pred) ** 0.5
    ],
    "R2": [
        r2_score(y_test, baseline_pred),
        r2_score(y_test, grid_pred)
    ]
})

display(comparison)
